# 06 · Motion

Motion from a video of one or more people, played on the avatar. The stage finds the people, follows each with SAM2, fits the avatar's own body to each person frame by frame, and writes a clip the runtime and the viewer play: bone rotations relative to the bind pose, root motion and face expressions, one track per person.

Run it from the shell (`go-task motion -- path/to/video.mp4`), or set `RUN` below. Results go to `outputs/motion/<name>/`.

Colours: one per person, in the order they were found.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Video
from PIL import Image

from avatar_splats.body_models.anny import AnnyBody
from avatar_splats.config import load_config
from avatar_splats.data import read_frames, write_video
from avatar_splats.export import AvatarRuntime, exported_path, read_avatar
from avatar_splats.motion import (
    MotionClip,
    MotionFit,
    Observations,
    extract_motion,
    fitted_people,
    motion_path,
)
from avatar_splats.motion.preview import clip_camera, render_clip
from avatar_splats.rendering import rasterize

config = load_config("../configs/default.yaml")
device = config.motion.device

VIDEO = Path("../examples/motion_reference-2.mp4")  # the motion video
NAME = VIDEO.stem
RUN = True  # True: extract the motion here (a few minutes); False: read an earlier run
REFIT = False  # with RUN: fit again from the saved observations, skipping detection

if RUN:
    extract_motion(config, VIDEO, NAME, refit=REFIT)
observations = Observations.load(motion_path(config, NAME, "observations"))
fit = MotionFit.load(motion_path(config, NAME, "fit"))
clip = MotionClip.load(motion_path(config, NAME, "clip"))
frames = list(read_frames(VIDEO, step=config.motion.frame_step))
COLOURS = np.array([(230, 80, 60), (60, 140, 230), (90, 190, 90), (220, 180, 40)])
print(f"{observations.num_people} people, {len(frames)} frames at {observations.fps:.1f} fps")

## Who was found

Each person's mask and keypoints on a few frames. Masks come from SAM2, which follows each person from the frame where all were detected; a mask that jumps to someone else shows up here first.

In [ ]:
shown = np.linspace(0, len(frames) - 1, 6).round().astype(int)
fig, axes = plt.subplots(1, len(shown), figsize=(20, 6))
masks = observations.mask_frames(shown.tolist())
scale = observations.mask_size[0] / observations.image_size[0]
for ax, (i, f) in zip(axes, enumerate(shown), strict=True):
    size = tuple(int(v) for v in observations.mask_size)
    image = np.array(Image.fromarray(frames[f]).resize(size)).astype(float)
    for p in range(observations.num_people):
        mask = masks[p, i]
        image[mask] = 0.5 * image[mask] + 0.5 * COLOURS[p % len(COLOURS)]
    ax.imshow(image.astype(np.uint8))
    for p in range(observations.num_people):
        points = observations.keypoints[p, f]
        seen = points[:, 2] >= 0.3
        ax.scatter(
            *(scale * points[seen, :2]).T, s=8, color=COLOURS[p % len(COLOURS)] / 255, edgecolor="k"
        )
    ax.set_title(f"frame {observations.frame_indices[f]}")
    ax.axis("off")
fig.tight_layout()

## Per-frame fit

The consistency diagnostic: keypoint error in pixels and silhouette IoU per person and frame. The IoU only counts pixels outside other people's masks, where a person can be seen. Grey bands mark frames where a person's keypoints were not found; their pose is only interpolated.

In [ ]:
fig, (error_ax, iou_ax) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for p in range(observations.num_people):
    colour = COLOURS[p % len(COLOURS)] / 255
    error_ax.plot(fit.frame_indices, fit.keypoint_error[p], color=colour, label=f"person {p}")
    iou_ax.plot(fit.frame_indices, fit.silhouette_iou[p], color=colour)
    for f in np.flatnonzero(~fit.seen[p]):
        for ax in (error_ax, iou_ax):
            ax.axvspan(fit.frame_indices[f] - 0.5, fit.frame_indices[f] + 0.5, color="0.9")
error_ax.set_ylabel("keypoint error (pixels)")
iou_ax.set_ylabel("silhouette IoU")
iou_ax.set_xlabel("frame")
error_ax.legend()
fig.tight_layout()
print("median keypoint error:", np.round(np.median(fit.keypoint_error, axis=1), 1), "px")
print("median silhouette IoU:", np.round(np.median(fit.silhouette_iou, axis=1), 3))

## The fitted bodies over the video

Each person's fitted body, drawn in their colour over the frame. Nearer bodies hide farther ones, so who is in front can be checked too.

In [ ]:
body = AnnyBody().to(device)
camera = fit.camera().scaled(0.4).to(device)
vertices = fitted_people(fit, body, shown.tolist())
fig, axes = plt.subplots(1, len(shown), figsize=(20, 6))
for ax, (i, f) in zip(axes, enumerate(shown), strict=True):
    screen = torch.cat([camera.screen(v) for v in vertices[:, i]])
    count = vertices.shape[2]
    faces = torch.cat([body.faces + p * count for p in range(len(vertices))])
    owner = torch.arange(len(vertices), device=device).repeat_interleave(len(body.faces))
    fragments = rasterize(screen, faces, camera.width, camera.height)
    person = fragments.image(owner[fragments.faces, None].float(), fill=-1)[..., 0]
    person = person.round().long().cpu().numpy()
    drawn = person >= 0
    image = np.array(Image.fromarray(frames[f]).resize((camera.width, camera.height))).astype(float)
    image[drawn] = 0.4 * image[drawn] + 0.6 * COLOURS[person[drawn] % len(COLOURS)]
    ax.imshow(image.astype(np.uint8))
    ax.axis("off")
fig.tight_layout()

## The clip

Seen from above: where each person's root goes over the clip. The clip's world is levelled (the floor under the lowest points is z = 0) and centred on the people at the first frame. The camera's place is marked.

In [ ]:
plt.figure(figsize=(6, 6))
for p in range(clip.num_people):
    path = clip.root_translation[p]
    plt.plot(*path[:, :2].T, color=COLOURS[p % len(COLOURS)] / 255, label=f"person {p}")
    plt.scatter(*path[0, :2], color=COLOURS[p % len(COLOURS)] / 255, marker="o")
plt.scatter(*clip.camera_to_world[:2, 3], marker="^", color="k", label="camera")
plt.axis("equal")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.legend();

The face's strongest expressions, as the avatar's ICT-FaceKit weights. They come from MediaPipe's ARKit blendshapes on a crop around each face, so they are only as good as the face is large in the frame.

In [ ]:
fig, axes = plt.subplots(clip.num_people, 1, figsize=(12, 3 * clip.num_people), squeeze=False)
for p, ax in enumerate(axes[:, 0]):
    strongest = np.argsort(-clip.expressions[p].max(axis=0))[:6]
    for e in strongest:
        ax.plot(fit.frame_indices, clip.expressions[p, :, e], label=clip.expression_names[e])
    ax.set_title(f"person {p}", loc="left")
    ax.set_ylabel("weight")
    ax.legend(ncol=3, fontsize=8)
axes[-1, 0].set_xlabel("frame")
fig.tight_layout()

## The avatar performing it

The exported avatar, one copy per person, played by the reference runtime (`export/runtime.py`) from the video's camera: the video, the two over each other, and the avatar alone.

In [ ]:
runtime = AvatarRuntime(read_avatar(exported_path(config))).to(device)
background = torch.ones(3, device=device)
camera = clip_camera(clip, fit.camera().scaled(0.4).to(device))
renders = list(render_clip(runtime, clip, camera, background, frames=shown.tolist()))
tiles = []
for f, render in zip(shown, renders, strict=True):
    video = np.array(Image.fromarray(frames[f]).resize(render.shape[1::-1]))
    tiles.append(np.concatenate([video, (0.5 * video + 0.5 * render).astype(np.uint8), render]))
plt.figure(figsize=(20, 14))
plt.imshow(np.concatenate(tiles, axis=1))
plt.axis("off");

The whole clip as a video, the source beside the avatar, written to `outputs/motion/<name>/preview.mp4`.

In [ ]:
preview = motion_path(config, NAME, "clip").with_name("preview.mp4")
renders = render_clip(runtime, clip, camera, background)
write_video(
    preview,
    (
        np.concatenate([np.array(Image.fromarray(frame).resize(r.shape[1::-1])), r], axis=1)
        for frame, r in zip(frames, renders, strict=True)
    ),
    clip.fps,
)
Video(str(preview), embed=True, width=900)

From another viewpoint: what one camera could not see, such as how far apart the people are, shows from the side.

In [ ]:
# The root sits at the hips: look at the middle of where the people's hips go.
target = torch.tensor(clip.root_translation.reshape(-1, 3).mean(axis=0), device=device).float()
side = camera.looking_at(target + torch.tensor([3.5, -1.0, 0.3], device=device), target)
renders = list(render_clip(runtime, clip, side, background, frames=shown.tolist()))
plt.figure(figsize=(20, 5))
plt.imshow(np.concatenate(renders, axis=1))
plt.axis("off");